Running on Colab, will download and install neccesary packages.
upload the "newBenchmarks_dataset.xlsx" in the main folder

In [ ]:
!pip install tensorboardX
!pip install pysr

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.5/87.5 kB 1.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.5/54.5 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 99.3/99.3 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.3/253.3 kB 11.3 MB/s eta 0:00:00


In [ ]:
!git clone https://github.com/AndreasMadsen/stable-nalu.git

Cloning into 'stable-nalu'...
remote: Enumerating objects: 3003, done.
remote: Counting objects: 100% (111/111), done.
remote: Compressing objects: 100% (56/56), done.
remote: Total 3003 (delta 57), reused 109 (delta 55), pack-reused 2892 (from 1)
Receiving objects: 100% (3003/3003), 14.82 MiB | 19.99 MiB/s, done.
Resolving deltas: 100% (2271/2271), done.


In [ ]:
cd stable-nalu

/content/stable-nalu


In [ ]:
!python setup.py develop

/usr/local/lib/python3.12/dist-packages/setuptools/_distutils/dist.py:261: UserWarning: Unknown distribution option: 'test_suite'
  warnings.warn(msg)
/usr/local/lib/python3.12/dist-packages/setuptools/_distutils/dist.py:261: UserWarning: Unknown distribution option: 'tests_require'
  warnings.warn(msg)
running develop
/usr/local/lib/python3.12/dist-packages/setuptools/command/develop.py:41: EasyInstallDeprecationWarning: easy_install command is deprecated.
!!

        ********************************************************************************
        Please avoid running ``setup.py`` and ``easy_install``.
        Instead, use pypa/build, pypa/installer or other
        standards-based tools.

        See https://github.com/pypa/setuptools/issues/917 for details.
        ********************************************************************************

!!
  easy_install.initialize_options(self)
/usr/local/lib/python3.12/dist-packages/setuptools/_distutils/cmd.py:66: SetuptoolsDepre

In [ ]:
import numpy as np
import pandas as pd
import pickle

In [ ]:
import torch
import torch.nn as nn
from stable_nalu.layer.re_regualized_linear_nac import ReRegualizedLinearNACLayer
from stable_nalu.layer.re_regualized_linear_nalu import ReRegualizedLinearNALULayer

/content/stable-nalu/stable_nalu/functional/gumbel.py:26: SyntaxWarning: invalid escape sequence '\i'
  tau: the temperature used, must be tau \in (0, \infty]. tau < 1
/content/stable-nalu/stable_nalu/layer/nac.py:13: SyntaxWarning: invalid escape sequence '\h'
  Asumming \hat{w} and \hat{m} are sampled from a uniform


In [ ]:
SEED=8
rnd=np.random.default_rng(0)
torch.manual_seed(SEED)

In [ ]:
!ls ../

models	sample_data  stable-nalu  test.xlsx


In [ ]:
data= pd.read_excel(r"../newBenchmarks_dataset.xlsx","rydberg")

In [ ]:
data.shape

(50, 14)

load the data for training. the data is collected by running the original paper.

In [ ]:
unique_train_indexes = list(rnd.choice(data.shape[0], 40, replace=False,))
unique_train_indexes

[np.int64(40),
 np.int64(23),
 np.int64(28),
 np.int64(45),
 np.int64(38),
 np.int64(46),
 np.int64(17),
 np.int64(43),
 np.int64(39),
 np.int64(3),
 np.int64(0),
 np.int64(13),
 np.int64(44),
 np.int64(41),
 np.int64(6),
 np.int64(24),
 np.int64(1),
 np.int64(37),
 np.int64(42),
 np.int64(21),
 np.int64(14),
 np.int64(30),
 np.int64(25),
 np.int64(16),
 np.int64(34),
 np.int64(8),
 np.int64(36),
 np.int64(11),
 np.int64(20),
 np.int64(22),
 np.int64(29),
 np.int64(18),
 np.int64(31),
 np.int64(15),
 np.int64(33),
 np.int64(47),
 np.int64(4),
 np.int64(9),
 np.int64(7),
 np.int64(26)]

In [ ]:
unique_test_indexes = [_ for _ in range(data.shape[0]) if _ not in unique_train_indexes]
len(unique_train_indexes),len(unique_test_indexes)

(40, 10)

In [ ]:
rydberg_train_X = data.loc[unique_train_indexes,['n1','n2']].values
rydberg_train_y = data.loc[unique_train_indexes,['y']].values

rydberg_test_X = torch.from_numpy(data.loc[unique_test_indexes,['n1','n2']].values)
rydberg_test_y = torch.from_numpy(data.loc[unique_test_indexes,['y']].values)


In [ ]:
rydberg_train_X.shape,rydberg_train_y.shape,rydberg_test_X.shape,rydberg_test_y.shape

((40, 2), (40, 1), torch.Size([10, 2]), torch.Size([10, 1]))

In [ ]:
from torch.utils.data import Dataset, DataLoader
loader = DataLoader(list(zip(rydberg_train_X,rydberg_train_y)), shuffle=True, batch_size=5)

In [ ]:
kwargs_nalu={'eps': 1e-07,'nac_oob': 'clip','regualizer_shape': 'linear','regualizer_z': 0,'mnac_epsilon': 0,'nalu_bias': False,'nalu_two_nac': True,'nalu_two_gate': False,'nalu_mul': 'mnac','nalu_gate': 'normal'}
kwargs_nac={'eps': 1e-07,'nac_oob': 'clip','regualizer_shape': 'linear','regualizer_z': 0,'mnac_epsilon': 0,'nalu_bias': False,'nalu_two_nac': False,'nalu_two_gate': False,'nalu_mul': 'normal','nalu_gate': 'normal'}


In [ ]:
class MyModel(nn.Module):
    def __init__(self, in_feats,h_feats, out_feat):
        super(MyModel, self).__init__()
        self.net0=nn.Linear(in_feats , h_feats*2,bias=False)
        self.net1=nn.Linear(h_feats*2 , h_feats,bias=False)
        self.leaKyrelu=torch.nn.LeakyReLU(0.1)
        self.net_nac=ReRegualizedLinearNACLayer(in_features=h_feats,out_features=h_feats,**kwargs_nac)
        self.net_nalu=ReRegualizedLinearNALULayer(in_features=h_feats,out_features=out_feat,**kwargs_nalu)
    def forward(self,inputs_):
        h=self.net0(inputs_.float())
        h=self.leaKyrelu(h)
        h=self.net1(h)
        h=self.leaKyrelu(h)
        h=self.net_nac(h)
        h=self.net_nalu(h)
        return h.double()
    def reset_parameters(self):
        torch.nn.init.xavier_uniform(self.net0.weight)
        torch.nn.init.xavier_uniform(self.net1.weight)
        self.net_nalu.reset_parameters()
        self.net_nac.reset_parameters()
    def regualizer(self):
        self.net_nalu.regualizer()
        self.net_nac.regualizer()

In [ ]:
net=MyModel(2,4,1)
net.reset_parameters()

/tmp/ipykernel_9338/2466979556.py:18: FutureWarning: `nn.init.xavier_uniform` is now deprecated in favor of `nn.init.xavier_uniform_`.
  torch.nn.init.xavier_uniform(self.net0.weight)
/tmp/ipykernel_9338/2466979556.py:19: FutureWarning: `nn.init.xavier_uniform` is now deprecated in favor of `nn.init.xavier_uniform_`.
  torch.nn.init.xavier_uniform(self.net1.weight)


In [ ]:
optim = torch.optim.RMSprop(net.parameters(), lr=0.001, weight_decay=1e-5)

In [ ]:
best_loss=np.inf
criterion = torch.nn.MSELoss()
for i in range(120000):
    loss_list=[]
    # net.train()
    for X_t,y_t in loader:
        optim.zero_grad()
        out=net(X_t)
        loss_train_criterion = criterion(y_t,out)
#         loss_train_regualizer =10 * r_w_scale * regualizers['W'] + regualizers['g'] + 0 * regualizers['z'] + 1 * regualizers['W-OOB']
        loss = loss_train_criterion #+ loss_train_regualizer
        loss.retain_grad()
        loss_list.append(loss.item())
        #################
        mea = torch.mean(torch.abs(y_t - out))
        loss.backward()
        optim.step()
    if i%1==0:
        with torch.no_grad():
            test_loss=criterion(rydberg_test_y,net(rydberg_test_X)).item()
            if test_loss<best_loss:
                best_loss=test_loss
                torch.save(net,f'../models/modelrydberg_2inputs_{i}.pt')

                print(f'saved at {i} with error {np.array(loss_list).mean()}, {best_loss}')

saved at 0 with error 187.8870893686519, 191.0186580638381
saved at 1 with error 181.81978199471726, 182.94734185069348
saved at 2 with error 168.46457047306038, 170.99838341792747
saved at 3 with error 152.9895325837299, 158.69610029157644
saved at 4 with error 138.01040768729808, 146.79415867920778
saved at 5 with error 123.80413960357407, 135.29225876123533
saved at 6 with error 110.22897353907503, 124.25500530483137
saved at 7 with error 97.30827310050427, 113.77162547663806
saved at 8 with error 85.13586116356294, 103.97395004479351
saved at 9 with error 73.84446224667168, 94.99322717621737
saved at 10 with error 63.5630181525481, 86.93018421514681
saved at 11 with error 54.40172543102166, 79.84947784374671
saved at 12 with error 46.42237726523108, 73.78577463883536
saved at 13 with error 39.61249000113932, 68.69493763172599
saved at 14 with error 33.97632952042923, 64.55967902509977
saved at 15 with error 29.436845013434112, 61.32844764140756
saved at 16 with error 25.93012350952

In [ ]:
i

119999

load best model saved for measuring the performance on all datasets

In [ ]:
net1=torch.load('../models/modelrydberg_2inputs_119704.pt',weights_only=False)#99842

In [ ]:
res_=[]
for n1,n2 in ((1,5),(2,4),(1,8),(2,8),(3,8),(4,8),(5,8),(6,8),(7,8),(1,9),(2,9),(3,9),(4,9),(5,9),(6,9),(7,9),(8,9),(1,10),(2,10),(3,10),(4,10),(5,10),(6,10),(7,10),(8,10),(9,10)):
    m=net1(torch.from_numpy(np.array([[n1,n2]])))
    obs=np.log(1/(1.10E+07*((1/n1**2)-(1/n2**2))))
    res_.append([n1,n2,m.detach().numpy()[0][0],obs])

In [ ]:
res_ = pd.DataFrame(res_)
res_

,0,1,2,3
0,1,5,-16.136345,-16.172584
1,2,4,-14.621179,-14.539429
2,1,8,-16.164640,-16.197657
3,2,8,-14.780425,-14.762573
4,3,8,-13.398439,-13.864631
5,4,8,-13.094972,-13.153135
6,5,8,-12.694060,-12.499209
7,6,8,-11.879654,-11.803208
8,7,8,-10.984447,-10.870753
9,1,9,-16.153603,-16.200983


In [ ]:
(res_[3] - res_[2]).abs().mean()

np.float64(0.2642748706907793)

In [ ]:
n1=10
n2=11
m=net1(torch.from_numpy(np.array([[n1,n2]])))
obs=np.log(1/(1.10E+07*((1/n1**2)-(1/n2**2))))
print(m.detach().numpy()[0][0],obs)

-10.417257308959961 -9.856967536901236


In [ ]:
pd.DataFrame(np.concatenate([data.loc[:,['n1','n2']].values,net1(torch.from_numpy(data.loc[:,['n1','n2']].values)).detach().numpy()],axis=1))

,0,1,2
0,4.0,5.0,-12.340358
1,4.0,7.0,-12.944950
2,6.0,7.0,-11.342027
3,4.0,6.0,-12.735641
4,6.0,7.0,-11.342027
5,2.0,3.0,-14.326780
6,2.0,5.0,-14.570326
7,2.0,7.0,-14.717483
8,1.0,6.0,-16.155441
9,3.0,5.0,-13.426346
